# Vintage Car Classifier - Training on GPU (all 196 classes)

Run this notebook in Google Colab with **GPU runtime** enabled.

Runtime → Change runtime type → T4 GPU

After training, download the zip and place files in your local project.

In [ ]:
# Install dependencies
!pip install -q torch torchvision matplotlib seaborn scikit-learn pyyaml pillow tqdm kagglehub scipy

In [ ]:
import os, sys, json, time, random, shutil
from collections import defaultdict

import numpy as np
from scipy.io import loadmat
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Download Stanford Cars dataset via kagglehub
import kagglehub
path = kagglehub.dataset_download('eduardo4jesus/stanford-cars-dataset')
print(f'Downloaded to: {path}')
KAGGLE_CACHE = path

In [ ]:
# Load class names and annotations
meta = loadmat(os.path.join(KAGGLE_CACHE, 'car_devkit', 'devkit', 'cars_meta.mat'))
class_names = [name[0] for name in meta['class_names'][0]]
print(f'Total classes: {len(class_names)}')

train_ann = loadmat(os.path.join(KAGGLE_CACHE, 'car_devkit', 'devkit', 'cars_train_annos.mat'))
train_items = []
for item in train_ann['annotations'][0]:
    train_items.append((str(item['fname'][0]), int(item['class'][0,0]) - 1))

with open(os.path.join(KAGGLE_CACHE, 'car_devkit', 'devkit', 'train_perfect_preds.txt')) as f:
    test_labels = [int(line.strip()) - 1 for line in f.read().strip().splitlines()]

test_dir = os.path.join(KAGGLE_CACHE, 'cars_test', 'cars_test')
test_fnames = sorted([f for f in os.listdir(test_dir) if f.endswith('.jpg')])
test_items = list(zip(test_fnames, test_labels[:len(test_fnames)]))

all_ann = train_items + test_items
print(f'Total images: {len(all_ann)}')

In [ ]:
# Organize all 196 classes into train/val/test folders
BASE = '/content/cars_data'
if os.path.exists(BASE):
    shutil.rmtree(BASE)

for split in ['train', 'val', 'test']:
    for i, name in enumerate(class_names):
        safe = name.replace('/', '_').replace(':', '_').replace(' ', '_')
        os.makedirs(os.path.join(BASE, split, safe), exist_ok=True)

def find_img(fname):
    for d in [
        os.path.join(KAGGLE_CACHE, 'cars_train', fname),
        os.path.join(KAGGLE_CACHE, 'cars_train', 'cars_train', fname),
        os.path.join(KAGGLE_CACHE, 'cars_test', fname),
        os.path.join(KAGGLE_CACHE, 'cars_test', 'cars_test', fname),
    ]:
        if os.path.exists(d):
            return d
    return None

random.seed(42)
image_map = defaultdict(list)
for fname, cls in all_ann:
    image_map[cls].append(fname)

copied = 0
for cls, files in image_map.items():
    safe = class_names[cls].replace('/', '_').replace(':', '_').replace(' ', '_')
    random.shuffle(files)
    n = len(files)
    splits = [
        ('train', files[:int(n*0.7)]),
        ('val', files[int(n*0.7):int(n*0.85)]),
        ('test', files[int(n*0.85):]),
    ]
    for split, split_files in splits:
        dst_dir = os.path.join(BASE, split, safe)
        for f in split_files:
            src = find_img(f)
            if src:
                shutil.copy2(src, os.path.join(dst_dir, os.path.basename(f)))
                copied += 1

print(f'Copied {copied} images across {len(image_map)} classes')

In [ ]:
# Build EfficientNet-B0 with 196 classes
num_classes = len(class_names)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, num_classes),
)
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EfficientNet-B0')
print(f'Classes: {num_classes}')
print(f'Params: {total_params:,} total, {trainable:,} trainable')

In [ ]:
# Data loaders with augmentation
IMG_SIZE = 224
BATCH_SIZE = 64
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    normalize,
])
eval_tfm = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    normalize,
])

train_ds = datasets.ImageFolder(os.path.join(BASE, 'train'), transform=train_tfm)
val_ds = datasets.ImageFolder(os.path.join(BASE, 'val'), transform=eval_tfm)
test_ds = datasets.ImageFolder(os.path.join(BASE, 'test'), transform=eval_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} images')
print(f'Batches per epoch: train={len(train_loader)}, val={len(val_loader)}')

In [ ]:
# Training loop with ETA
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
PATIENCE = 7
best_loss = float('inf')
best_acc = 0.0
no_improve = 0
start_time = time.time()

def fmt(s):
    h, m = int(s // 3600), int((s % 3600) // 60)
    return f'{h}h{m}m' if h else f'{m}m{int(s%60)}s'

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for imgs, labs in train_loader:
        imgs, labs = imgs.to(device), labs.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        train_total += labs.size(0)
        train_correct += pred.eq(labs).sum().item()

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labs in val_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            out = model(imgs)
            loss = criterion(out, labs)
            val_loss += loss.item() * imgs.size(0)
            _, pred = out.max(1)
            val_total += labs.size(0)
            val_correct += pred.eq(labs).sum().item()

    scheduler.step(val_loss / val_total)
    epoch_time = time.time() - epoch_start
    elapsed = time.time() - start_time
    avg_epoch = elapsed / epoch
    eta = avg_epoch * (EPOCHS - epoch)

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    train_l = train_loss / train_total
    val_l = val_loss / val_total

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Train: {train_l:.3f} / {train_acc:.1f}% | '
          f'Val:   {val_l:.3f} / {val_acc:.1f}% | '
          f'{fmt(epoch_time)} | ETA: {fmt(eta)}', flush=True)

    if val_l < best_loss:
        best_loss = val_l
        best_acc = val_acc
        no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_l,
            'val_acc': val_acc,
            'class_names': train_ds.classes,
        }, '/content/best_model.pth')
        print(f'  -> Saved checkpoint ({val_acc:.1f}%)', flush=True)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping after {epoch} epochs', flush=True)
            break

In [ ]:
# Test the best model
checkpoint = torch.load('/content/best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labs in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        _, pred = out.max(1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labs.numpy())

test_acc = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f'Test Accuracy: {test_acc:.2f}%')

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=train_ds.classes, yticklabels=train_ds.classes)
plt.title('Confusion Matrix - 196 classes')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
print('Saved confusion_matrix.png')

In [ ]:
# Export to TorchScript and save metadata
model.eval()
example = torch.randn(1, 3, 224, 224).to(device)
traced = torch.jit.trace(model, example)
traced.save('/content/model_torchscript.pt')
print('Exported model_torchscript.pt')

meta = {
    'class_names': train_ds.classes,
    'num_classes': len(train_ds.classes),
    'test_accuracy': round(test_acc, 2),
}
with open('/content/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved metadata.json')

In [ ]:
# Zip everything and download
!zip -j /content/training_output.zip \
  /content/best_model.pth \
  /content/model_torchscript.pt \
  /content/confusion_matrix.png \
  /content/metadata.json

from google.colab import files
files.download('/content/training_output.zip')